# HTML UI Agent - Fine-tuning with Unsloth

Qwen2.5-Coder-7B 모델을 UI HTML 생성용으로 파인튜닝합니다.

**런타임 설정**: `런타임` → `런타임 유형 변경` → **GPU** 선택 (T4 이상)

## 1. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
%%capture
# Unsloth 설치 (약 2-3분 소요)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. 학습 데이터 업로드

왼쪽 파일 탐색기에서 `training_data.jsonl` 파일을 업로드하세요.

In [ ]:
# Google Drive 마운트 (대용량 데이터 사용 시)
# from google.colab import drive
# drive.mount('/content/drive')

# 또는 직접 업로드
from google.colab import files
uploaded = files.upload()  # training_data.jsonl 선택

In [ ]:
# 데이터 확인
import json

TRAINING_DATA_PATH = "training_data.jsonl"  # 업로드한 파일명으로 변경

# 샘플 확인
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        item = json.loads(line)
        print(f"=== Sample {i+1} ===")
        print(f"User: {item['messages'][1]['content'][:100]}...")
        print(f"Assistant: {item['messages'][2]['content'][:100]}...")
        print()

# 총 개수
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    total = sum(1 for _ in f)
print(f"총 학습 데이터: {total}개")

## 3. 모델 로드

In [ ]:
from unsloth import FastLanguageModel
import torch

# 설정
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

# 모델 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("✅ 모델 로드 완료!")

In [ ]:
# LoRA 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## 4. 데이터셋 준비

In [ ]:
from datasets import Dataset

# 데이터 로드
data = []
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"데이터 로드: {len(data)}개")

# 포맷팅
def format_prompt(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Dataset 생성
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

# Train/Val 분리
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"✅ Train: {len(train_dataset)}, Validation: {len(eval_dataset)}")

## 5. 학습 실행

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "./qwen-html-ui-lora"

# 학습 설정
# T4: batch_size=2, A100: batch_size=4
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # T4용, A100은 4
    gradient_accumulation_steps=8,   # 효과적 배치=16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

In [ ]:
# 학습 시작 (T4 기준 약 2-4시간 소요)
print("🚀 학습 시작...")
trainer.train()

In [ ]:
# 모델 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 모델 저장 완료: {OUTPUT_DIR}")

## 6. 추론 테스트

In [ ]:
# 추론 모드로 전환
FastLanguageModel.for_inference(model)

# 테스트 프롬프트
test_prompts = [
    "로그인 페이지 만들어줘",
    "검색 기능이 있는 헤더 영역 필요해",
    "저장 버튼과 닫기 버튼이 있는 하단 영역 만들어줘",
]

system_prompt = """당신은 HTML/CSS UI 전문가입니다. 사용자의 요청에 따라 기존 디자인 시스템과 일관된 HTML 코드를 생성합니다.
- 시맨틱 HTML5 태그를 사용합니다
- 클래스명은 기존 컴포넌트 라이브러리의 규칙을 따릅니다
- 접근성(a11y)을 고려합니다
- 깔끔하고 유지보수하기 쉬운 코드를 작성합니다"""

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"📝 요청: {prompt}")
    print(f"{'='*60}")
    print(response.split("assistant")[-1].strip())

## 7. GGUF 변환 (로컬 실행용)

In [ ]:
# GGUF로 변환 (Q4_K_M 양자화)
# 로컬 CPU에서 실행하기 위한 형식

GGUF_DIR = "qwen-html-ui-gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"  # 4bit 양자화
)

print(f"✅ GGUF 파일 생성: {GGUF_DIR}/unsloth.Q4_K_M.gguf")

In [ ]:
# Ollama용 Modelfile 생성
modelfile = '''FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 2048

SYSTEM """당신은 HTML/CSS UI 전문가입니다. 사용자의 요청에 따라 기존 디자인 시스템과 일관된 HTML 코드를 생성합니다.
- 시맨틱 HTML5 태그를 사용합니다
- 클래스명은 기존 컴포넌트 라이브러리의 규칙을 따릅니다
- 접근성(a11y)을 고려합니다
- 깔끔하고 유지보수하기 쉬운 코드를 작성합니다"""
'''

with open(f"{GGUF_DIR}/Modelfile", "w") as f:
    f.write(modelfile)

print("✅ Modelfile 생성 완료!")

In [ ]:
# 파일 다운로드
!zip -r qwen-html-ui-gguf.zip {GGUF_DIR}/

from google.colab import files
files.download("qwen-html-ui-gguf.zip")

## 8. 로컬에서 사용하기

다운로드한 zip 파일을 로컬에서:

```bash
# 1. 압축 해제
unzip qwen-html-ui-gguf.zip

# 2. Ollama에 모델 등록
cd qwen-html-ui-gguf
ollama create html-ui-agent -f Modelfile

# 3. 실행
ollama run html-ui-agent
```